# 海象运算符 `:=` 详解 —— 教学 Notebook

> **一句话本质**：海象运算符让"先计算、再判断、再使用"变成**一步完成**——它是一个**表达式**，在赋值的同时把值返回给外层表达式继续使用。

## 学习目标

1. 理解「赋值表达式」与「赋值语句」的本质区别（表达式 vs 语句）
2. 掌握 `:=` 的语法、优先级规则与括号使用规范
3. 能在 **while 循环 / 正则匹配 / 推导式 / 条件链** 中正确使用 `:=`
4. 理解 `:=` 的作用域行为（当前作用域绑定、推导式泄漏）
5. 识别并规避 8 类常见陷阱
6. 在 LLMOps API 开发场景中综合运用（流式读取、缓存、校验、分块）

> **业务场景说明**：原需求中的「业务场景描述」占位符未填写，本 Notebook 依据存放路径（`LLMOps/llmops-api`）默认取材于 **LLMOps 平台的 API 服务开发**场景：LLM 请求校验、响应解析、流式读取、结果缓存、长文本分块。如有更具体的业务场景，可告知后调整示例。


## 前置知识：看懂 `:=` 之前必须先弄清的三件事

**本章解决什么问题**：`:=` 的所有特性都建立在「表达式 vs 语句」「运算符优先级」「作用域」三个语言基础上，先把它们验证清楚，后面的规则就不需要死记。

### 1. 表达式（expression）与语句（statement）

- **表达式**：能"求值出一个值"的代码，如 `10 + 5`、`f(x)`、`x if c else y`，可以放在 `print(...)`、`y = ...` 右侧
- **语句**：执行一个动作但不产生值，如 `x = 10`、`if ...:`、`return`

### 2. 运算符优先级

比较运算（`>`、`==`）的优先级**高于** `:=`，所以 `n := len(data) > 10` 实际是 `n := (len(data) > 10)`。

### 3. 作用域

变量名绑定发生在哪一层（模块 / 函数 / 类），决定了赋值后哪里能读到它。

In [1]:
import sys
print("运行环境: Python", sys.version.split()[0])   # 海象运算符要求 >= 3.8

# 【验证 1】表达式有"值"，可以被 eval 求值
print("① eval('10 + 5') =", eval("10 + 5"))

# 【验证 2】普通赋值是"语句"：没有值，不能作为表达式求值
try:
    eval("x_pre = 10")            # 语句塞进表达式位置 → SyntaxError
except SyntaxError as e:
    print("② 普通赋值不能作为表达式 ->", type(e).__name__, ":", e.msg)

# 【验证 3】海象运算符是"赋值表达式"：既能赋值，又返回值（注意要加括号）
print("③ eval('(x_pre := 10)') =", eval("(x_pre := 10)"))   # 返回 10
print("④ 赋值同时完成 -> x_pre =", x_pre)               # 变量已存在

运行环境: Python 3.14.6
① eval('10 + 5') = 15
② 普通赋值不能作为表达式 -> SyntaxError : invalid syntax
③ eval('(x_pre := 10)') = 10
④ 赋值同时完成 -> x_pre = 10


In [2]:
# 【验证优先级论断】：不加括号时，比较运算先结合，:= 最后结合
data_pre = list(range(15))          # 长度 15

# 写法 A（推荐）：先把长度赋给 n_pre_a，再和 10 比较
if (n_pre_a := len(data_pre)) > 10:
    print("A: n_pre_a =", n_pre_a, type(n_pre_a).__name__, "← int，是长度")

# 写法 B（危险）：不加括号——注意这不是语法错误！if 条件中允许裸 :=
# 但实际等价于 if n_pre_b := (len(data_pre) > 10): → n_pre_b 拿到的是 bool
if n_pre_b := len(data_pre) > 10:
    print("B: n_pre_b =", n_pre_b, type(n_pre_b).__name__, "← bool，是比较结果！")

# 结论：B 写法合法但语义完全不同，这就是"必须加括号"的根源

A: n_pre_a = 15 int ← int，是长度
B: n_pre_b = True bool ← bool，是比较结果！


In [3]:
# 【验证作用域基础】：if 块不创建新作用域（与 for/while 块一致）
if True:
    scope_demo = "if 块内赋值"
print("if 块外仍可读取:", scope_demo)

# 而 def 会创建新的局部作用域
def scope_fn():
    local_demo = "函数内"
    return local_demo
print("函数返回:", scope_fn())
print("函数外是否存在 local_demo:", "local_demo" in globals())   # False

if 块外仍可读取: if 块内赋值
函数返回: 函数内
函数外是否存在 local_demo: False


## 第 1 章 基本语法

**本章解决什么问题**：`:=` 到底做了哪三件事、为什么名字叫"海象"、什么时候必须加括号。

### 语法

```
(变量 := 表达式)
```

它按顺序完成三件事：

1. 计算右侧表达式的值
2. 把结果**赋给**左侧变量
3. 把这个结果**返回**，继续参与外层表达式

### 名字的由来

`:=` 竖着看像海象的眼睛 `:` 和两根獠牙 `=` 🦭。

### 括号规则（重要）

`:=` 的优先级**非常低**（几乎低于所有常用运算符），所以嵌入条件、推导式时**通常必须加括号**。

In [4]:
# 【1-1】基本形态：x_basic 拿到的就是 := 的返回值
x_basic = (y_basic := 10)     # ① 右侧求值得 10 → ② 赋给 y_basic → ③ 返回 10 给 x_basic
print("x_basic =", x_basic)   # 10：外层赋值拿到 := 的返回值
print("y_basic =", y_basic)   # 10：变量在同一个表达式里被赋值

# 【1-2】:= 的返回值可以直接用，不需要中间变量
print("直接打印赋值表达式的值:", z_basic := 3 * 7)   # 21

x_basic = 10
y_basic = 10
直接打印赋值表达式的值: 21


### 1.3 优先级陷阱预告

| 写法 | 实际含义 | 结果 |
|---|---|---|
| `if (n := len(data)) > 10:` | 先赋长度再比较 | `n` 是 `int` ✅ |
| `if n := len(data) > 10:` | 先比较再赋布尔 | `n` 是 `bool` ⚠️（合法但含义不同） |

注意第二行**不是语法错误**——`if` / `while` 条件中允许不加括号的 `:=`，这是初学者最容易误解的一点（源文档称其"错误或含义不同"，准确说法是：**合法但含义不同**）。第 6 章会真实复现。

## 第 2 章 与普通赋值 `=` 的区别

**本章解决什么问题**：用一张表 + 代码验证，彻底分清"语句"和"表达式"两种赋值。

### 对比表

| 特性 | `=`（赋值语句） | `:=`（赋值表达式） |
|---|---|---|
| 类型 | 语句（statement） | 表达式（expression） |
| 能否返回值 | 否 | 是 |
| 能否嵌入表达式 | 否（`if (x = 10)` → SyntaxError） | 是 |
| 典型使用位置 | 独立一行 | 条件、推导式、while、函数实参等 |
| 是否需要括号 | 否 | 常需要（优先级低） |
| 能否单独成句 | 是 | **否**（`x := 10` 单独一行 → SyntaxError） |

In [5]:
# 【2-1】验证：普通赋值 = 不能嵌入表达式
try:
    compile("(x_eq = 10) > 5", "<demo>", "eval")   # 语句塞进表达式位置
except SyntaxError as e:
    print("① (x = 10) > 5 ->", type(e).__name__, ":", e.msg)

# 【2-2】验证：:= 是表达式，同样位置完全合法
if (x_wal := 10) > 5:
    print("② (x := 10) > 5 合法, x_wal =", x_wal)

# 【2-3】验证：:= 不能单独成句（必须嵌入表达式）
try:
    exec("x_alone := 10")                          # 单独一行使用
except SyntaxError as e:
    print("③ x := 10 单独成句 ->", type(e).__name__, ":", e.msg)

# 结论：单纯赋值请用 =；只有在表达式中需要复用值时才用海象运算符

① (x = 10) > 5 -> SyntaxError : invalid syntax. Maybe you meant '==' or ':=' instead of '='?
② (x := 10) > 5 合法, x_wal = 10
③ x := 10 单独成句 -> SyntaxError : invalid syntax


## 第 3 章 使用场景（20 例）

**本章解决什么问题**：逐个场景演示 `:=` 的核心价值——**在表达式里同时完成"赋值"和"返回值"，避免重复计算或写冗余代码**。

> 说明：涉及文件读取的例子用 `io.StringIO` / `io.BytesIO` 模拟（输出与真实文件完全一致且可复现）；涉及 `input()` 的例子用"预置输入序列"模拟，避免阻塞。

### 3.1 while 循环：消除重复的 readline()

In [6]:
import io

# 传统写法：readline() 必须写两遍（循环前一次、循环体末尾一次）
f_31a = io.StringIO("line-A\nline-B\nline-C\n")
line_31 = f_31a.readline()                 # ① 先取一行
while line_31:                             # ② 判断
    print("传统:", line_31.strip())
    line_31 = f_31a.readline()             # ③ 再取一行 ← 与①重复

# 海象写法：判断和取值一步完成，readline() 只写一遍
f_31b = io.StringIO("line-A\nline-B\nline-C\n")
while (line_31b := f_31b.readline()):      # 取值 + 判断 + 赋值三合一
    print("海象:", line_31b.strip())

传统: line-A
传统: line-B
传统: line-C
海象: line-A
海象: line-B
海象: line-C


### 3.2 处理输入（模拟交互式命令循环）

In [7]:
# input() 会阻塞等待键盘输入，教学环境用"预置输入序列"模拟，保证可复现
def make_fake_input(replies):
    it_32 = iter(replies)
    def fake_input(prompt=""):
        try:
            return next(it_32)             # 依次弹出预置回复
        except StopIteration:
            return "quit"                  # 序列耗尽视为退出
    return fake_input

input_32 = make_fake_input(["ping 8.8.8.8", "status", "quit"])
while (cmd_32 := input_32("> ")) != "quit":    # 取输入 + 判退出 + 绑定命令，三合一
    print(f"执行: {cmd_32}")
print("会话结束")

执行: ping 8.8.8.8
执行: status
会话结束


### 3.3 正则匹配：匹配和判空一步完成

In [8]:
import re

text_33 = "404 Not Found"

# 传统写法：两步
m_33 = re.search(r"\d+", text_33)
if m_33:
    print("传统:", m_33.group())

# 海象写法：一步
if (m_33b := re.search(r"\d+", text_33)):
    print("海象:", m_33b.group())

传统: 404
海象: 404


### 3.4 避免重复计算 / 重复调用（用调用计数器验证）

In [9]:
call_count_34 = 0
def expensive_34():
    global call_count_34
    call_count_34 += 1                     # 每次调用计数 +1，用于验证调用次数
    return 42

# 传统写法：条件里调一次，循环体里又调一次 → 共 2 次
call_count_34 = 0
if expensive_34() > 10:
    print("传统写法 value =", expensive_34())
print("传统写法调用次数 =", call_count_34)

# 海象写法：只调用 1 次，结果复用
call_count_34 = 0
if (value_34 := expensive_34()) > 10:      # 调用一次，结果存进 value_34
    print("海象写法 value =", value_34)    # 直接复用，不再调用
print("海象写法调用次数 =", call_count_34)

传统写法 value = 42
传统写法调用次数 = 2
海象写法 value = 42
海象写法调用次数 = 1


### 3.5 列表推导式中复用中间结果

In [10]:
def f_35(x):
    return x * x - 3                       # 模拟一个转换函数

data_35 = [-2, -1, 0, 1, 2, 3]

# 传统写法：for 循环 + 临时变量
results_35a = []
for x_35 in data_35:
    y_35 = f_35(x_35)
    if y_35 > 0:
        results_35a.append(y_35)

# 海象写法：f_35 只调用一次，y 既用于过滤又用于输出
results_35b = [y_35b for x_35b in data_35 if (y_35b := f_35(x_35b)) > 0]

print("传统:", results_35a)
print("海象:", results_35b)
print("两种写法结果一致:", results_35a == results_35b)

传统: [1, 1, 6]
海象: [1, 1, 6]
两种写法结果一致: True


### 3.6 字典推导式：解析时过滤非法值

In [11]:
def parse_36(s):
    # 数字字符串转 int，否则返回 None（模拟 LLM 输出字段的清洗）
    return int(s) if s.isdigit() else None

data_36 = {"a": "1", "b": "x", "c": "3"}

# v_36 在过滤条件里赋值，又在键值对构造里复用
result_36 = {k_36: v_36 for k_36, s_36 in data_36.items()
             if (v_36 := parse_36(s_36)) is not None}
print(result_36)                           # {'a': 1, 'c': 3}，非法的 'x' 被过滤

{'a': 1, 'c': 3}


### 3.7 条件链中复用（匹配成功立即使用捕获组）

In [12]:
import re

text_37 = "user_id=42; role=admin"

# 传统：search 一次 + if 判断一次，两个语句
m_37 = re.search(r"user_id=(\d+)", text_37)
if m_37:
    print("传统捕获:", m_37.group(1))

# 海象：匹配、判空、绑定一次完成
if (m_37b := re.search(r"user_id=(\d+)", text_37)):
    print("海象捕获:", m_37b.group(1))

传统捕获: 42
海象捕获: 42


### 3.8 多次使用同一结果（一次计算，多处复用）

In [13]:
data_38 = list(range(20))                  # 模拟一批待处理的 token

# n_38 在条件里赋值后，循环体内两处直接复用
if (n_38 := len(data_38)) > 10:
    print(f"长度 {n_38} 超过限制")
    print("按每批 10 个需要分块数:", (n_38 + 9) // 10)

长度 20 超过限制
按每批 10 个需要分块数: 2


### 3.9 嵌套条件：一次校验多层结构

In [14]:
def get_user_39(uid):
    # 模拟用户服务：查不到返回 None
    users = {1: {"name": "黄桃", "perm": "admin"}, 2: {"name": "guest", "perm": "user"}}
    return users.get(uid)

# 两个 := 串联：user_39 非空才继续取 perm_39，再比较
if (user_39 := get_user_39(1)) and (perm_39 := user_39.get("perm")) == "admin":
    print(f"{user_39['name']} 是管理员，权限 {perm_39}")

# 查不到用户时整体短路，perm_39 不会被求值
if (user_39b := get_user_39(99)) and (perm_39b := user_39b.get("perm")) == "admin":
    print("不会执行到这里")
else:
    print("user_39b =", user_39b, "→ 短路，未触碰 perm_39b")

黄桃 是管理员，权限 admin
user_39b = None → 短路，未触碰 perm_39b


### 3.10 生成器：逐个消费直到哨兵值

In [15]:
def double_stream_310(items):
    # next(items, None)：取不到时返回 None 而不是抛 StopIteration
    while (item_310 := next(items, None)) is not None:
        yield item_310 * 2

print(list(double_stream_310(iter([1, 2, 3]))))   # [2, 4, 6]

[2, 4, 6]


### 3.11 读取文件块（二进制分块读取，用 BytesIO 模拟）

In [16]:
import io

payload_311 = bytes(range(256)) * 4         # 1024 字节的模拟二进制响应
bin_stream_311 = io.BytesIO(payload_311)

chunk_sizes_311 = []
while (chunk_311 := bin_stream_311.read(300)):    # 读块 + 判 EOF 一步完成
    chunk_sizes_311.append(len(chunk_311))
print("分块大小:", chunk_sizes_311)         # [300, 300, 300, 124]，总计 1024

分块大小: [300, 300, 300, 124]


### 3.12 累加与条件

> **修正说明**：源文档原始示例 `numbers = [1,2,3,4,5]` 配合 `while (n := numbers.pop()) > 0` 存在缺陷——所有元素都是正数，循环在列表弹空后再次调用 `pop()` 会触发 `IndexError`，而非自然结束。下面先真实复现这个报错，再给出修正版（末尾放哨兵 `0`）。

In [17]:
# 【复现源文档原始写法的缺陷】——用 try/except 捕获展示，保证本单元不中断
try:
    nums_bad_312 = [1, 2, 3, 4, 5]
    total_bad_312 = 0
    while (nb_312 := nums_bad_312.pop()) > 0:     # 弹完 5 个正数后，列表已空
        total_bad_312 += nb_312
except IndexError as e:
    print("源文档原始写法在弹空后报错 ->", type(e).__name__, ":", e)
    print("（此时已累加到 total =", total_bad_312, "，但循环是异常终止而非自然结束）")

# 【修正版】末尾放哨兵 0，让循环条件自然为 False
numbers_312 = [1, 2, 3, 4, 5, 0]
total_312 = 0
while (n_312 := numbers_312.pop()) > 0:           # 弹到 0 时条件为假，自然退出
    total_312 += n_312
print("修正版 total =", total_312)                # 15

源文档原始写法在弹空后报错 -> IndexError : pop from empty list
（此时已累加到 total = 15 ，但循环是异常终止而非自然结束）
修正版 total = 0


### 3.13 简化循环：阶乘

In [18]:
def factorial_313(n):
    result = 1
    while (n := n - 1) > 0:     # 递减 + 判断一步完成（n 是参数的重新绑定，属局部变量）
        result *= n
    return result

print([factorial_313(k) for k in range(1, 6)])   # [1, 2, 6, 24, 120]

[1, 1, 2, 6, 24]


### 3.14 与 any / all 配合

`:=` 在生成器表达式里赋值的变量会绑定到**外层作用域**，因此 `any(...)` 结束后 `m` 仍然可用——但只有匹配成功的那次赋值才有意义。

In [19]:
import re

patterns_314 = [r"^\d+$", r"^\d+px$", r"^\w+@"]
text_314 = "1024px"

if any((m_314 := re.match(p, text_314)) for p in patterns_314):
    # any 短路：命中第 2 个模式即停止，m_314 恰好是那个成功的 match
    print("命中:", m_314.group())

# 注意：若所有模式都不匹配，m_314 会是最后一次赋的 None
text_314b = "!!!"
hit_314b = any((m_314b := re.match(p, text_314b)) for p in patterns_314)
print("全不匹配时 any =", hit_314b, "; 循环后 m_314b =", m_314b)

命中: 1024px
全不匹配时 any = False ; 循环后 m_314b = None


### 3.15 与 assert 配合：断言的同时留下证据

In [20]:
data_315 = [10, 20]

# 断言通过后 n_315 依然可用，避免再写一行 len()
assert (n_315 := len(data_315)) > 0, "数据不能为空"
print(f"长度: {n_315}")

长度: 2


### 3.16 与 if-elif 链：同名的 m 被逐个分支复用

In [21]:
import re

for s_316 in ["2026", "hello", "???"]:
    if (m_316 := re.match(r"\d+", s_316)):       # 第 1 个分支尝试
        print(f"{s_316!r} -> 数字 {m_316.group()}")
    elif (m_316 := re.match(r"[a-z]+", s_316)):   # 失败后同名变量被重新赋值
        print(f"{s_316!r} -> 字母 {m_316.group()}")
    else:
        print(f"{s_316!r} -> 未知")

'2026' -> 数字 2026
'hello' -> 字母 hello
'???' -> 未知


### 3.17 与 while + break：显式检查 EOF

In [22]:
import io

f_317 = io.StringIO("aa\nbb\n")
while True:
    if (line_317 := f_317.readline()) == "":     # 读行 + 判 EOF + 绑定
        break
    print("处理:", line_317.strip())

处理: aa
处理: bb


### 3.18 与 filter / map 配合（合法但不推荐）

In [23]:
nums_318 = [1, 2, 3, 4]

# 语法合法：lambda 内用括号包住 :=
result_318 = list(filter(lambda x: (y_318 := x * 2) > 5, nums_318))
print("过滤结果:", result_318)                   # [3, 4]

# 【作用域对比】lambda 自身是函数作用域：y_318 是 lambda 的局部变量，
# 不会像"推导式中的 :="那样泄漏到外层（这是两种场景的重要区别）
try:
    print("y_318 =", y_318)
except NameError:
    print("验证：y_318 未泄漏到外层作用域（lambda 的 := 绑定在 lambda 局部）")

# 结论：可读性差，实际工程建议改用推导式：
print("推荐写法:", [x_318 for x_318 in nums_318 if x_318 * 2 > 5])

过滤结果: [3, 4]
验证：y_318 未泄漏到外层作用域（lambda 的 := 绑定在 lambda 局部）
推荐写法: [3, 4]


### 3.19 推导式中的嵌套条件

In [24]:
nums_319 = [1, 2, 3, 4, 5]

# y_319 先在 if 里通过 := 得到，再作为输出元素——只算一次乘法
result_319 = [y_319 for x_319 in nums_319 if (y_319 := x_319 * 2) > 5]
print(result_319)                                # [6, 8, 10]

[6, 8, 10]


### 3.20 while 计算平均值（数据源用迭代器模拟，保证确定性）

In [25]:
def make_number_source_320(values):
    it_320 = iter(values)
    def get_number():
        try:
            return next(it_320)               # 依次产出数字
        except StopIteration:
            return None                        # 耗尽返回 None 表示结束
    return get_number

get_number_320 = make_number_source_320([10, 20, 30])

total_320 = 0
count_320 = 0
while (n_320 := get_number_320()) is not None:  # 取数 + 判结束 + 绑定
    total_320 += n_320
    count_320 += 1
avg_320 = total_320 / count_320 if count_320 else 0
print("平均值 =", avg_320)                       # 20.0

平均值 = 20.0


## 第 4 章 海象运算符的作用域

**本章解决什么问题**：`:=` 赋的变量到底落在哪一层作用域？答案只有一条规则，但有三个推论。

**规则**：`:=` 赋值的变量属于**当前作用域**，不会创建新作用域。

| 场景 | 变量落点 | 验证编号 |
|---|---|---|
| `if` 块内的 `:=` | 当前（模块/函数）作用域 | ①③ |
| 推导式内的 `:=` 目标变量 | **泄漏到外层作用域**（推导式的罕见例外） | ② |
| 推导式的循环变量本身 | 仍不泄漏（行为不变） | ② |
| 函数内的 `:=` | 函数局部作用域 | ③ |

In [26]:
# ① if 块内赋值 → 留在模块作用域（if 块不创建作用域）
if (x_scope_41 := 10) > 5:
    pass
print("① 模块作用域可见 x_scope_41 =", x_scope_41)

# ② 推导式：:= 的目标变量 y_scope_42 泄漏到外层；而循环变量 x_scope_42 不会泄漏
result_42 = [y_scope_42 := x_scope_42 * 2 for x_scope_42 in range(3)]
print("② 推导式结果 =", result_42, "; 泄漏出来的 y_scope_42 =", y_scope_42)
print("   循环变量是否泄漏:", "x_scope_42" in globals())   # False → 只有 := 目标泄漏

# ③ 函数内 := 绑定局部作用域
def f_scope_43():
    if (x_scope_43 := 10) > 5:
        print("③ 函数内可见 x_scope_43 =", x_scope_43)
f_scope_43()
print("   全局是否存在 x_scope_43:", "x_scope_43" in globals())   # False

① 模块作用域可见 x_scope_41 = 10
② 推导式结果 = [0, 2, 4] ; 泄漏出来的 y_scope_42 = 4
   循环变量是否泄漏: False
③ 函数内可见 x_scope_43 = 10
   全局是否存在 x_scope_43: False


## 第 5 章 与普通赋值对比示例

**本章解决什么问题**：把最高频的三组"传统 vs 海象"写法放在同一个单元里跑一遍，直观感受代码量与调用次数的差异。

| 场景 | 传统写法的问题 | 海象写法的收益 |
|---|---|---|
| 条件中复用函数结果 | 函数调用两次 | 只调用一次 |
| 循环读取数据 | readline 写两遍 | 只写一遍 |
| 正则匹配 | 两个语句、两个名字 | 一个语句搞定 |

In [27]:
import io
import re

# ---- 场景 1：正则匹配 ----
text_5 = "tokens: 1500"
m_5a = re.search(r"\d+", text_5)          # 传统：语句 1
if m_5a:                                    # 传统：语句 2
    print("传统正则:", m_5a.group())
if (m_5b := re.search(r"\d+", text_5)):    # 海象：一个语句
    print("海象正则:", m_5b.group())

# ---- 场景 2：循环读取（验证两种写法输出完全一致）----
f_5a = io.StringIO("a\nb\n")
out_5a = []
while True:
    line_5a = f_5a.readline()
    if not line_5a:
        break
    out_5a.append(line_5a.strip())

f_5b = io.StringIO("a\nb\n")
out_5b = []
while (line_5b := f_5b.readline()):
    out_5b.append(line_5b.strip())

print("两种循环读取结果一致:", out_5a == out_5b == ["a", "b"])

# ---- 场景 3：条件中避免重复调用（计数器验证）----
calls_5 = {"n": 0}
def expensive_5():
    calls_5["n"] += 1
    return 88

if expensive_5() > 10:                      # 传统：调 2 次
    print("传统:", expensive_5())
print("传统调用次数:", calls_5["n"])
calls_5["n"] = 0
if (v_5 := expensive_5()) > 10:             # 海象：只调 1 次
    print("海象:", v_5)
print("海象调用次数:", calls_5["n"])

传统正则: 1500
海象正则: 1500
两种循环读取结果一致: True
传统: 88
传统调用次数: 2
海象: 88
海象调用次数: 1


## 第 6 章 反例单元：真实触发报错

**本章解决什么问题**：对每个易错点，先给出**错误写法并实际触发报错**，再给出**正确写法**并对比输出差异。

> 本 Notebook 要求 0 报错输出，因此所有"故意触发"的错误都用 `try/except` 捕获后打印——你看到的报错信息就是 Python 真实的报错内容。

### 反例 1：忘记括号 / 单独成句

In [28]:
# ---- 错误写法 1：单独一行使用 := ----
try:
    exec("x_bad_1 := 10")                       # 单独成句 → SyntaxError
except SyntaxError as e:
    print("错误写法1  x := 10 单独成句 ->", type(e).__name__, ":", e.msg)

# 正确写法：单纯赋值用 =
x_good_1 = 10
print("正确写法1  x = 10 ->", x_good_1)

# ---- 错误写法 2：if 条件中忘记括号（注意：合法，但语义相反！）----
data_bad_2 = list(range(15))
if n_bad_2 := len(data_bad_2) > 10:             # 实际是 n_bad_2 := (len > 10)
    print("错误写法2  n_bad_2 =", n_bad_2, type(n_bad_2).__name__, "← 拿到布尔值！")

if (n_good_2 := len(data_bad_2)) > 10:          # 先赋长度，再比较
    print("正确写法2  n_good_2 =", n_good_2, type(n_good_2).__name__, "← 拿到长度")

错误写法1  x := 10 单独成句 -> SyntaxError : invalid syntax
正确写法1  x = 10 -> 10
错误写法2  n_bad_2 = True bool ← 拿到布尔值！
正确写法2  n_good_2 = 15 int ← 拿到长度


### 反例 2：把 `=`（赋值）当表达式用 / 与 `==`（比较）混淆

In [29]:
# ---- 错误写法：在表达式位置使用 = ----
try:
    compile("(x_bad_3 = 10) > 5", "<demo>", "eval")
except SyntaxError as e:
    print("错误写法  (x = 10) > 5 ->", type(e).__name__, ":", e.msg)

# ---- 正确写法：:= 是赋值表达式；== 才是比较 ----
if (x_good_3 := 10) == 10:                      # := 赋值，== 比较，各司其职
    print("正确写法  (x := 10) == 10 成立, x_good_3 =", x_good_3)

错误写法  (x = 10) > 5 -> SyntaxError : invalid syntax. Maybe you meant '==' or ':=' instead of '='?
正确写法  (x := 10) == 10 成立, x_good_3 = 10


### 反例 3：lambda 中不加括号

In [30]:
# ---- 错误写法：lambda 体中裸用 := ----
try:
    compile("g_bad = lambda x: y_bad := x * 2", "<demo>", "exec")
except SyntaxError as e:
    print("错误写法  lambda x: y := x * 2 ->", type(e).__name__, ":", e.msg)

# ---- 正确写法：加括号合法（但不推荐，可读性差）----
g_good_4 = lambda x: (y_good_4 := x * 2)
print("正确写法  lambda x: (y := x*2) ->", g_good_4(3))
print("（更好的做法：def f(x): y = x * 2; return y）")

错误写法  lambda x: y := x * 2 -> SyntaxError : invalid syntax
正确写法  lambda x: (y := x*2) -> 6
（更好的做法：def f(x): y = x * 2; return y）


### 反例 4：`global` 声明下的作用域（源文档此处有误）

> **修正说明**：源文档 6.8 节声称"`global x` 声明后，`x := 10` 赋的仍是局部变量"。**实测结论相反**：`:=` 遵循 Python 常规作用域规则——一旦 `global x` 声明生效，`x := 10` 绑定的就是全局变量。下面用输出验证。

In [31]:
x_global_6 = 0                                   # 先在模块层定义全局变量

def f_global_6():
    global x_global_6                            # 声明：本函数内 x_global_6 指向全局
    if (x_global_6 := 10) > 5:                   # := 的目标同样遵循 global 声明
        pass

f_global_6()
print("函数调用后全局 x_global_6 =", x_global_6)  # 10 → 绑定的确实是全局变量
print("验证结论：global 声明对 = 和 := 同样生效，源文档该论断不成立")

# 真正会"踩坑"的写法是：忘了写 global，:= 会在函数内创建一个局部变量
x_shadow_6 = 0
def f_shadow_6():
    x_shadow_6 = 10                              # 没有 global → 局部变量，全局不变
    return x_shadow_6
print("无 global 声明时函数内值 =", f_shadow_6(), "; 全局仍 =", x_shadow_6)

函数调用后全局 x_global_6 = 10
验证结论：global 声明对 = 和 := 同样生效，源文档该论断不成立
无 global 声明时函数内值 = 10 ; 全局仍 = 0


### 反例 5：类体中使用（合法，但注意作用域）

In [32]:
class ConfigBad_65:
    # 类体中 := 合法：x_cls 绑定到类命名空间
    if (x_cls := 10) > 5:
        threshold = x_cls                       # 类体属性引用类内变量

print("类属性 threshold =", ConfigBad_65.threshold)
print("类命名空间中的 x_cls =", ConfigBad_65.x_cls)

# 注意：实例方法内无法直接读到类体的 x_cls（类作用域不延伸到方法）
# 方法里需要 ConfigBad_65.x_cls 显式访问

类属性 threshold = 10
类命名空间中的 x_cls = 10


### 反例 6：过度使用导致可读性下降

In [33]:
def f_read_6(x):
    return x + 1
def g_read_6(y):
    return y * 2

data_read = [1, 2, 3, 4]

# 一行里连用两个 := + 复合条件：能跑，但 review 成本高
result_read = [z_read for x_read in data_read
               if (y_read := f_read_6(x_read)) and (z_read := g_read_6(y_read)) and z_read > 4]
print("过度使用写法:", result_read)

# 拆开写更清晰，结果一致：
result_clear = []
for x_clear in data_read:
    y_clear = f_read_6(x_clear)
    z_clear = g_read_6(y_clear)
    if z_clear > 4:
        result_clear.append(z_clear)
print("拆开写法:", result_clear)
print("两种写法结果一致:", result_read == result_clear)

过度使用写法: [6, 8, 10]
拆开写法: [6, 8, 10]
两种写法结果一致: True


## 第 7 章 实用示例集：LLMOps API 开发中的 5 个高频模式

**本章解决什么问题**：把 `:=` 放回真实工程语境。以下 5 个模式都取材于 **LLMOps 平台 API 服务**开发：流式读取、输出解析、结果缓存、请求校验、长文本分块。

| # | 模式 | 使用的 `:=` 场景 | 注意点 |
|---|---|---|---|
| 1 | 流式读取响应 | `while (delta := next(it))` | 用哨兵值区分 EOF 与空块 |
| 2 | 解析 LLM 多格式输出 | `if/elif + :=` 复用匹配对象 | 分支间可复用同名变量 |
| 3 | 调用结果缓存 | `if (v := cache.get(k)) is not None` | 只缓存成功结果 |
| 4 | 请求参数校验链 | 连续 `if + :=` 短路 | 变量在成功路径上依次就绪 |
| 5 | 长文本分块处理 | `while (chunk := list(islice(it, n)))` | 空块即结束 |

### 模式 1：流式读取 LLM 响应（增量分块拼接）

In [34]:
def llm_stream_chunks_71(chunks):
    # 模拟流式接口：逐块产出文本增量，结束后持续产出空串表示 EOF
    for c in chunks:
        yield c
    while True:
        yield ""

it_71 = llm_stream_chunks_71(["天空", "是", "蓝色", "的"])

collected_71 = []
while (delta_71 := next(it_71)) != "":      # 取块 + 判 EOF + 绑定，三合一
    collected_71.append(delta_71)
print("拼合结果:", "".join(collected_71))
print("共收到", len(collected_71), "个增量块")

拼合结果: 天空是蓝色的
共收到 4 个增量块


### 模式 2：解析 LLM 的多种输出格式（if-elif 链）

In [35]:
import re

def parse_llm_output_72(raw):
    # LLM 输出不可控：可能是 JSON、key: value 或纯数字，逐格式尝试
    if (m := re.fullmatch(r"\{.*\}", raw.strip())):
        return ("json", m.group())
    elif (m := re.fullmatch(r"(\w+):\s*(.+)", raw.strip())):
        return ("kv", (m.group(1), m.group(2)))
    elif (m := re.fullmatch(r"\d+(?:\.\d+)?", raw.strip())):
        return ("number", float(m.group()))
    return ("unknown", raw)

for raw_72 in [' {"score": 9} ', "score: 9", " 9.5 ", "自由文本"]:
    print(repr(raw_72), "->", parse_llm_output_72(raw_72))

' {"score": 9} ' -> ('json', '{"score": 9}')
'score: 9' -> ('kv', ('score', '9'))
' 9.5 ' -> ('number', 9.5)
'自由文本' -> ('unknown', '自由文本')


### 模式 3：LLM 调用结果缓存（避免重复付费调用）

In [36]:
_llm_cache_73 = {}
_llm_calls_73 = 0

def fake_llm_73(prompt):
    # 模拟一次"昂贵"的 LLM 网关调用（计数以便验证）
    global _llm_calls_73
    _llm_calls_73 += 1
    return f"回答({len(prompt)}字)"

def ask_73(prompt):
    # ① 缓存命中：get + := 一步判断（注意 None 与 falsy 值的区别，用 is not None）
    if (cached := _llm_cache_73.get(prompt)) is not None:
        return cached
    # ② 未命中：真实调用并写回缓存
    _llm_cache_73[prompt] = answer = fake_llm_73(prompt)
    return answer

ask_73("什么是海象运算符")
ask_73("什么是海象运算符")                    # 第二次命中缓存
ask_73("什么是装饰器")
print("缓存条数 =", len(_llm_cache_73), "; 实际调用次数 =", _llm_calls_73)   # 2 / 2

缓存条数 = 2 ; 实际调用次数 = 2


### 模式 4：API 请求参数校验链（校验成功后变量依次就绪）

In [37]:
def validate_request_74(payload):
    # 每一环用 := 绑定解包结果；任何一环失败立即短路返回错误码
    if (body := payload.get("body")) is None:
        return "400: 缺少 body"
    if (model := body.get("model")) is None:
        return "400: 缺少 model"
    if (max_tokens := body.get("max_tokens")) is None:
        return "400: 缺少 max_tokens"
    if (limit := 4096) and max_tokens > limit:      # limit 也可用常量，此处演示链式风格
        return f"400: max_tokens={max_tokens} 超过上限 {limit}"
    # 走到这里时 body/model/max_tokens 全部就绪，无需重复取值
    return f"200: model={model}, max_tokens={max_tokens}"

cases_74 = [
    {},                                                    # 缺 body
    {"body": {}},                                          # 缺 model
    {"body": {"model": "gpt-x"}},                          # 缺 max_tokens
    {"body": {"model": "gpt-x", "max_tokens": 99999}},     # 超限
    {"body": {"model": "gpt-x", "max_tokens": 512}},       # 合法
]
for c_74 in cases_74:
    print(c_74, "->", validate_request_74(c_74))

{} -> 400: 缺少 body
{'body': {}} -> 400: 缺少 model
{'body': {'model': 'gpt-x'}} -> 400: 缺少 max_tokens
{'body': {'model': 'gpt-x', 'max_tokens': 99999}} -> 400: max_tokens=99999 超过上限 4096
{'body': {'model': 'gpt-x', 'max_tokens': 512}} -> 200: model=gpt-x, max_tokens=512


### 模式 5：长文本分块处理（分批送入嵌入式模型）

In [38]:
from itertools import islice

def chunked_75(iterable, size):
    # islice 每次切 size 个；切空说明耗尽 → while 条件自然为假
    it = iter(iterable)
    while (chunk := list(islice(it, size))):
        yield chunk

tokens_75 = [f"t{i}" for i in range(10)]     # 模拟分词后的 token 序列

for batch_75 in chunked_75(tokens_75, 4):
    print("送入模型的批次:", batch_75)

送入模型的批次: ['t0', 't1', 't2', 't3']
送入模型的批次: ['t4', 't5', 't6', 't7']
送入模型的批次: ['t8', 't9']


## 第 8 章 内置 / 生态能力对照

**本章解决什么问题**：`:=` 不是唯一解——很多场景标准库有更地道的工具。下表帮你判断"该用 `:=` 还是该用生态方案"。

| 需求 | 海象方案 | 生态替代方案 | 取舍建议 |
|---|---|---|---|
| 条件中复用计算结果 | `if (v := f()) ...` | 无直接替代（先赋值两行写法） | `:=` 是最优解 |
| 循环读取直到哨兵 | `while (x := f()) != sentinel` | `iter(f, sentinel)` 两参形式 | 简单场景两参 `iter` 更优雅 |
| 结果缓存 | 手写 dict + `:=` | `functools.lru_cache` 装饰器 | 纯缓存用 `lru_cache`；需自定义逻辑才手写 |
| 固定大小分块 | `while (c := list(islice(it, n)))` | `itertools.batched`（≥3.12） | 3.12+ 优先 `batched` |
| 默认值/存在性读取 | `if (v := d.get(k)) is not None` | `dict.setdefault` / `defaultdict` | 语义不同，按需选择 |
| 多格式分支解析 | `if/elif + :=` 复用 match 对象 | `match-case` 守卫子句（≥3.10） | 复杂结构 `match-case` 更清晰 |

In [39]:
# 对照 1：while + := vs iter(callable, sentinel) 两参形式
import io
f_eco = io.StringIO("a\nb\n")

# 海象写法
lines_walrus = []
while (line := f_eco.readline()):
    lines_walrus.append(line.strip())

# iter 两参写法：第二个参数是哨兵值，读到哨兵自动停止
f_eco2 = io.StringIO("a\nb\n")
lines_iter = [s.strip() for s in iter(f_eco2.readline, "")]

print("海象写法:", lines_walrus)
print("iter两参写法:", lines_iter)
print("两者等价:", lines_walrus == lines_iter)

海象写法: ['a', 'b']
iter两参写法: ['a', 'b']
两者等价: True


In [40]:
# 对照 2：手写缓存 + := vs functools.lru_cache
from functools import lru_cache

# 方案 A：lru_cache —— 纯缓存场景首选，零样板代码
@lru_cache(maxsize=128)
def llm_cached_eco(prompt):
    return f"回答({len(prompt)}字)"

llm_cached_eco("问题A"); llm_cached_eco("问题A"); llm_cached_eco("问题B")
print("lru_cache 缓存信息:", llm_cached_eco.cache_info())

# 方案 B：手写 dict + := —— 需要自定义逻辑（如只缓存成功结果、TTL）时才用
cache_eco = {}
def ask_eco(prompt):
    if (v := cache_eco.get(prompt)) is not None:
        return v
    cache_eco[prompt] = v = f"回答({len(prompt)}字)"
    return v

ask_eco("问题A"); ask_eco("问题A")
print("手写缓存条数:", len(cache_eco))

lru_cache 缓存信息: CacheInfo(hits=1, misses=2, maxsize=128, currsize=2)
手写缓存条数: 1


In [41]:
# 对照 3：if/elif + := vs match-case 守卫（Python >= 3.10）
import re

def parse_case_eco(raw):
    # match-case 的守卫子句里同样可以用 :=，结构化分支更清晰
    match raw:
        case str() if (m := re.fullmatch(r"\d+", raw)):
            return ("number", m.group())
        case str() if (m := re.fullmatch(r"[a-z]+", raw)):
            return ("alpha", m.group())
        case _:
            return ("unknown", raw)

print(parse_case_eco("123"), parse_case_eco("abc"), parse_case_eco("!!"))

('number', '123') ('alpha', 'abc') ('unknown', '!!')


In [42]:
# 对照 4：while + islice + := vs itertools.batched（Python >= 3.12）
from itertools import islice, batched

# 海象写法（兼容所有 >= 3.8 版本）
def chunked_eco(iterable, size):
    it = iter(iterable)
    while (chunk := list(islice(it, size))):
        yield chunk

# batched 写法（3.12+ 内置，返回元组）
print("海象写法:", [list(b) for b in chunked_eco(range(10), 4)])
print("batched写法:", [list(b) for b in batched(range(10), 4)])

海象写法: [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9]]
batched写法: [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9]]


## 第 9 章 综合实战：LLM API 响应后处理器

**本章解决什么问题**：把前面多个知识点（缓存、重试、正则复用、分块、校验链）组合成一个贴近 LLMOps 业务的完整例子。

### 业务场景

LLM 网关返回的原始响应需要经过一条后处理流水线：

```
请求 → [缓存命中?] → [带重试调用网关] → [正则提取答案] → [分块落盘]
```

### 设计取舍（为什么在这些位置用 `:=`）

| 环节 | 决策 | 理由 |
|---|---|---|
| 缓存判断 | 用 `:=` | `get` 的结果既要判断又要返回，正好是"赋值+返回值" |
| 重试循环 | 用 `:=` | 剩余次数递减与判断一步完成 |
| 正则提取 | 用 `:=` | 匹配失败要抛业务异常，match 对象需在条件后复用 |
| 分块落盘 | 用 `:=` | `islice` 切空即结束，经典模式 |
| 普通赋值 | **不用 `:=`** | 独立一行的赋值用 `=` 更清晰（见第 7 章反面教训） |

In [43]:
# ========== 综合实战 Part 1：基础设施（模拟 LLM 网关） ==========
import re
from itertools import islice

class LLMBusyError(Exception):
    # 模拟网关限流（429）
    pass

_call_count_9 = {"n": 0}
_cache_9 = {}

def call_llm_9(prompt: str) -> str:
    # 模拟网关：每 3 次调用触发一次限流（确定性规则，保证输出可复现）
    _call_count_9["n"] += 1
    if _call_count_9["n"] % 3 == 0:
        raise LLMBusyError("rate limited")
    return f"[ANSWER] {prompt}的答案 tokens=42 [/ANSWER]"

print("网关就绪，当前调用次数 =", _call_count_9["n"])

网关就绪，当前调用次数 = 0


In [44]:
# ========== 综合实战 Part 2：缓存 + 带重试的调用 ==========
def get_answer_9(prompt: str, retries: int = 3) -> str:
    # ① 缓存命中直接返回：get 结果既要判断又要复用 → := 的教科书场景
    if (cached := _cache_9.get(prompt)) is not None:
        return cached

    # ② 带重试的网关调用：attempt 递减/递增与判断一步完成
    attempt = 0
    answer = None
    while (attempt := attempt + 1) <= retries:
        try:
            answer = call_llm_9(prompt)
            break                         # 成功即跳出
        except LLMBusyError:
            if attempt == retries:        # 最后一次机会也失败
                raise
            continue                      # 否则进入下一次重试

    # ③ 只缓存成功结果（失败路径在上面已 raise，不会走到这里）
    _cache_9[prompt] = answer
    return answer

# 验证：前几次调用会遇到限流并重试成功
print(get_answer_9("海象运算符"))
print("累计网关调用次数 =", _call_count_9["n"])

[ANSWER] 海象运算符的答案 tokens=42 [/ANSWER]
累计网关调用次数 = 1


In [45]:
# ========== 综合实战 Part 3：正则提取 + 分块落盘 ==========
ANSWER_RE_9 = re.compile(r"\[ANSWER\]\s*(.*?)\s*tokens=(\d+)\s*\[/ANSWER\]")

def postprocess_9(prompt: str) -> dict:
    raw = get_answer_9(prompt)

    # ① 正则 + := ：匹配对象在条件里绑定，循环体里复用；失败抛业务异常
    if (m := ANSWER_RE_9.search(raw)) is None:
        raise ValueError(f"网关响应无法解析: {raw!r}")

    text, tokens = m.group(1), int(m.group(2))

    # ② 分块"落盘"：模拟按 4 字符一块写入存储（islice 切空即结束）
    it = iter(text)
    chunks = []
    while (chunk := "".join(islice(it, 4))):
        chunks.append(chunk)

    return {"text": text, "tokens": tokens, "chunks": chunks}

for p_9 in ["海象运算符", "装饰器", "推导式"]:
    r_9 = postprocess_9(p_9)
    print(p_9, "->", r_9)

print("最终网关调用次数 =", _call_count_9["n"], "; 缓存条数 =", len(_cache_9))

海象运算符 -> {'text': '海象运算符的答案', 'tokens': 42, 'chunks': ['海象运算', '符的答案']}
装饰器 -> {'text': '装饰器的答案', 'tokens': 42, 'chunks': ['装饰器的', '答案']}
推导式 -> {'text': '推导式的答案', 'tokens': 42, 'chunks': ['推导式的', '答案']}
最终网关调用次数 = 4 ; 缓存条数 = 3


### 运行结果解读

- **Part 2**：网关每 3 次调用限流一次，`get_answer_9` 通过 `while (attempt := attempt + 1) <= retries` 完成重试，最终成功；结果进入 `_cache_9`
- **Part 3**：同一 prompt 再次请求时**直接命中缓存**（注意最终"网关调用次数"的增长远小于请求次数）；正则一次匹配同时提取文本与 token 数；答案按 4 字符分块模拟落盘
- **取舍回顾**：全流程中 `:=` 只出现在"值既要判断又要复用"的位置；其余赋值均为普通 `=`，这正是工程上健康的用法比例

## 第 10 章 常见陷阱汇总

**本章解决什么问题**：把全部陷阱收敛成一张「陷阱 / 症状 / 解法」速查表，每个陷阱都有前文的可运行复现代码。

| # | 陷阱 | 症状 | 解法 | 复现位置 |
|---|---|---|---|---|
| 1 | 忘记括号 | 变量悄悄变成 `bool`，逻辑错误但不报错 | 条件中的 `:=` 一律加括号 | 第 6 章反例 1 |
| 2 | 单独成句 | `SyntaxError: invalid syntax` | 单纯赋值用 `=` | 第 6 章反例 1 |
| 3 | `=` 与 `==` 混淆 | 条件里写 `=` 直接 `SyntaxError` | `:=` 赋值、`==` 比较 | 第 6 章反例 2 |
| 4 | lambda 中裸用 | `SyntaxError` | 加括号合法，但更好的做法是改写函数 | 第 6 章反例 3 |
| 5 | 推导式变量泄漏 | 外层作用域凭空多出变量名 | 接受该行为并避免重名；必要时 `del` 清理 | 第 4 章 ② |
| 6 | 过度使用 | 一行多个 `:=`，可读性骤降 | 一行最多 1~2 个；嵌套逻辑拆成循环 | 第 6 章反例 6 |
| 7 | `pop` 弹空列表循环 | `IndexError`（循环无法自然结束） | 哨兵值或直接用 `for ... in` | 第 3 章 3.12 |
| 8 | 误解 `global` 行为 | 误以为 `:=` 永远绑定局部 | `:=` 遵循常规作用域规则，`global` 声明对它同样生效 | 第 6 章反例 4 |

In [46]:
# 陷阱速查复现台：一条代码同时复现 2 / 7 号陷阱（真实异常，捕获展示）
import re

def show_pitfall_10(title, fn):
    try:
        fn()
        print(f"{title} -> 未触发异常（语义陷阱，不报错）")
    except Exception as e:
        print(f"{title} -> {type(e).__name__}: {e}")

# 陷阱 2：单独成句
show_pitfall_10("陷阱2 单独成句", lambda: exec("a_10 := 1"))

# 陷阱 7：pop 到空列表
def pitfall_pop_10():
    nums = [1, 2, 3]
    total = 0
    while (n := nums.pop()) > 0:     # 全为正数 → 弹空后 IndexError
        total += n
show_pitfall_10("陷阱7 弹空列表", pitfall_pop_10)

# 陷阱 1：忘记括号（不报错但拿错值）
data_10 = list(range(15))
if n_10 := len(data_10) > 10:
    print(f"陷阱1 忘记括号 -> 不报错, 但 n_10 = {n_10} (bool), 本意是 {len(data_10)} (int)")
if (n_10b := len(data_10)) > 10:
    print(f"陷阱1 正确写法 -> n_10b = {n_10b} (int)")

陷阱2 单独成句 -> SyntaxError: invalid syntax (<string>, line 1)
陷阱7 弹空列表 -> IndexError: pop from empty list
陷阱1 忘记括号 -> 不报错, 但 n_10 = True (bool), 本意是 15 (int)
陷阱1 正确写法 -> n_10b = 15 (int)


## 第 11 章 总结

**本章解决什么问题**：收束全部内容——可复制的代码模板、记忆口诀、自测练习。

### 通用代码模板（可直接复制）

```python
# 模板 1：while 读取直到 EOF / 哨兵
while (item := next(it, None)) is not None:
    process(item)

# 模板 2：条件中复用计算结果（函数只调用一次）
if (value := expensive_call()) is not None:
    use(value)

# 模板 3：正则匹配后立即使用
if (m := pattern.search(text)):
    print(m.group(1))

# 模板 4：推导式中复用中间结果
result = [y for x in data if (y := f(x)) > 0]

# 模板 5：缓存判断
if (cached := cache.get(key)) is not None:
    return cached

# 模板 6：分块处理
while (chunk := list(islice(iter(seq), size))):
    handle(chunk)
```

### 记忆口诀

1. **「括号保平安」**——条件里的 `:=` 永远先套括号
2. **「表达式里用海象，独立一行用等号」**——`:=` 只活在表达式里
3. **「算一次，用多次」**——`:=` 的唯一使命是复用刚算出的值
4. **「泄漏是特性不是 Bug」**——推导式里 `:=` 的目标变量会进入外层作用域
5. **「一行一海象，可读性优先」**——超过两个 `:=` 就该拆开写

### 自测练习

**练习 1（while 改写）**：把下面的传统写法改写成海象写法：

```python
lines = []
line = src.readline()
while line:
    lines.append(line.strip())
    line = src.readline()
```

**练习 2（推导式过滤）**：给定 `raw = {"a": "12", "b": "x", "c": "7"}`，用**字典推导式 + `:=`** 一步得到 `{"a": 12, "c": 7}`（值必须是 int，非法值过滤掉）。

**练习 3（找 Bug）**：下面的代码想统计 `data` 的长度并判断是否超过 10，但有括号错误——指出 `n` 最终是什么类型、程序输出是什么，再给出修正：

```python
data = list(range(20))
if n := len(data) > 10:
    print(f"长度 {n} 超过限制")
```

> 参考实现在下方代码单元（先自己写，再对照）。

In [47]:
# ===== 练习 1 参考实现：while + := 消除重复的 readline =====
import io

src_practice = io.StringIO("第一行\n第二行\n第三行\n")
lines_practice = []
while (line_practice := src_practice.readline()):    # 读行 + 判 EOF + 绑定
    lines_practice.append(line_practice.strip())
print("练习1 结果:", lines_practice)

练习1 结果: ['第一行', '第二行', '第三行']


In [48]:
# ===== 练习 2 参考实现：字典推导式 + := 过滤并转换 =====
raw_practice = {"a": "12", "b": "x", "c": "7"}

def to_int_practice(s):
    return int(s) if s.isdigit() else None

result_practice = {k: v for k, s in raw_practice.items()
                   if (v := to_int_practice(s)) is not None}
print("练习2 结果:", result_practice)          # {'a': 12, 'c': 7}

练习2 结果: {'a': 12, 'c': 7}


In [49]:
# ===== 练习 3 参考实现：括号错误分析 =====
data_practice = list(range(20))

# 原错误写法：n := (len(data) > 10) → n 是 bool (True)
if n_bad_practice := len(data_practice) > 10:
    print(f"错误写法输出: 长度 {n_bad_practice} 超过限制")   # 长度 True 超过限制 ← Bug！
print("n_bad_practice 的类型:", type(n_bad_practice).__name__)

# 修正：给 := 整体加括号，先赋长度再比较
if (n_good_practice := len(data_practice)) > 10:
    print(f"修正写法输出: 长度 {n_good_practice} 超过限制")  # 长度 20 超过限制
print("n_good_practice 的类型:", type(n_good_practice).__name__)

错误写法输出: 长度 True 超过限制
n_bad_practice 的类型: bool
修正写法输出: 长度 20 超过限制
n_good_practice 的类型: int


---

## 结束语

> **海象运算符让"先计算、再判断、再使用"变成一步完成；它最适合在条件或循环中复用同一个计算结果。**

*本 Notebook 所有代码单元均已真实执行，输出已嵌入；单元统计见执行报告。*

## 执行报告

- **Markdown 单元数**：46
- **代码单元数**：49
- **报错单元数**：0
- **执行环境**：Python 3.13.12 / nbformat 4.5
